# Fitting EMRI tracks with `fewtrax` + `parismc-jax`

**Keep the EMRI-Search detection statistic; replace the identification stage with the JAX framework.**

This notebook outlines the integration described in
`JAX-waveform/Notes/detection_integration_multimode_report.md` (Part 1). The
single-harmonic *detection statistic* of Speri, Tenorio, Chapman-Bird & Gerosa
(arXiv:2510.20891) is **retained unchanged** as the trigger and as the
secondary-mode vetting statistic. The *identification* stage — going from a
phenomenological frequency track to intrinsic parameters and a posterior — is
replaced by:

1. **`fewtrax`** — a JAX-native, differentiable, `vmap`-able EMRI trajectory,
   giving a smooth track-residual loss $L(\theta)$ and its gradients/Hessian.
2. **`parismc-jax` (PARIS)** — a parallel adaptive importance sampler that maps
   the *structurally multimodal* identification landscape into a basin registry.
3. **Iterative higher-mode refinement** — predicting companion harmonics,
   vetting them with the kept detection statistic, and **reweighting** the PARIS
   registry, with no re-run.

> Cells that need `fewtrax` flux data or `parismc` are guarded so the notebook
> runs end-to-end on the `emrisearch`-only parts and clearly marks the heavy
> stages as an outline.

### Pipeline

```
 STFT data ──► det_stat (KEPT) ──► anchor track f̂(t)        [Stage 1, external]
                                        │
            ┌───────────────────────────┴───────────────────────────┐
            │  fewtrax  L(θ)            parismc-jax  PARIS            │  [Stage 2]
            │  + Adam/L-BFGS + Fisher   → basin registry {θ̂_j, w_j}  │
            └───────────────────────────┬───────────────────────────┘
                                        │  higher-mode reweight  P→P·e^{−L₂/2}
                                        ▼
                              narrow prior → final PE          [Stage 3]
```


## 0. Imports and configuration

`emrisearch` provides the detection statistic and the synthetic-injection
harness. We enable 64-bit precision (required for the trajectory ODE and the
Fisher matrix).

In [ ]:
import os, sys, time
os.environ.setdefault("GPUBACKENDTOOLS_FORCE_BACKEND", "cuda12x")

import numpy as np
import matplotlib.pyplot as plt
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

from few.utils.constants import YRSID_SI

sys.path.append("../src/")
from emrisearch.search_utils import generate_emri_signal_and_sfts
from emrisearch.emri_utils import get_f_fdot_fddot_back
from emrisearch.jax_utils import det_stat as jax_det_stat, psd
from emrisearch.track_optimizer import (
    TrackOptimizer, TrackOptimizerJAX, estimate_plunge_time,
    get_default_mode_candidates, DEFAULT_BOUNDS,
)

print("JAX devices:", jax.devices())

## 1. Synthetic injection

We generate one EMRI: signal + coloured TDI noise, segmented into Short Fourier
Transforms (SFTs). The true parameters are
$\theta_{\rm true}=(M,\mu,a,T_{\rm plunge},e_f)$ (plus a fixed extrinsic
amplitude scale).

The SFTs $\tilde d_j^{(\alpha)}$ form the time–frequency data the detection
statistic acts on: each column $\alpha$ is the windowed FFT of one segment of
duration $T_{\rm sft}$.

In [ ]:
# True EMRI parameters: [M, mu, a, T_plunge, e_f, x0]
true_values = np.array([1e6, 10.0, 0.5, 0.2, 0.1, 1.0])

T_data  = 0.2     # years (short, for a fast demo)
T_sft   = 5e4     # seconds (~14 h)
deltaT  = 5.0     # seconds
snr_ref = 30.0

injection = generate_emri_signal_and_sfts(
    true_values, T_data, T_sft, deltaT, snr_ref, T_data
)
data_sfts = jnp.asarray(injection["data_sfts"])
t_obs     = injection["t_alpha"]          # SFT mid-times [s]
num_sfts  = injection["num_sfts"]
print(f"SFT array : {data_sfts.shape}  ({num_sfts} segments)")
print(f"Final SNR : {injection['snr_final']:.1f}")

## 2. The anchor track and the **detection statistic we keep**

Stage 1 delivers a single phenomenological track for the dominant azimuthal
harmonic $(m,k,n)=(2,0,0)$: the instantaneous GW frequency at each SFT mid-time,
$\hat f(t_\alpha)$, with resolution $\sigma_f\approx 1/T_{\rm sft}$.

The semi-coherent **detection statistic** (Tenorio–Gerosa Eq. 7) integrates the
matched Fresnel-kernel response of a chirping tone along that track. For each
segment $\alpha$,

$$
c_\alpha=\sum_{j=k_\alpha-P}^{k_\alpha+P}\tilde d_j^{*}\,
\frac{K(f_\alpha-j\Delta f,\dot f_\alpha,T_{\rm sft})}{S_n(j\Delta f)},
\qquad k_\alpha=\lfloor f_\alpha T_{\rm sft}\rfloor,
$$

and the statistic is the noise-weighted power summed over segments,

$$
\boxed{\;\Lambda=\sum_\alpha \frac{|c_\alpha|^2}{h_\alpha},\qquad
h_\alpha=\frac{T_{\rm sft}}{2\,S_n(f_\alpha)}\;}
$$

with $K$ the Fresnel kernel (matched filter for a linear chirp, differentiable
via a custom VJP) and $S_n$ the TDI PSD. **This function is unchanged from the
original pipeline** — `emrisearch.jax_utils.det_stat`. We use it twice: as the
detection trigger here, and to vet predicted secondary harmonics in §7.

In [ ]:
phi, f, dotf, dotdotf = injection["true_phi_f_fdot_fddot"]

# Dominant harmonic (m, n) = (2, 0): f_alpha = m f_phi + n f_r
m, n = 2, 0
f_obs    = m * f[0]    + n * f[1]      # Hz
fdot_obs = m * dotf[0] + n * dotf[1]   # Hz/s

stat_true = float(jax_det_stat(data_sfts, jnp.asarray(f_obs),
                               jnp.asarray(fdot_obs), T_sft=T_sft))
print(f"Detection statistic Lambda at the (2,0,0) anchor track: {stat_true:.1f}")

t_days = t_obs / 86400.0
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(t_days, f_obs * 1e3, "C0.-"); ax[0].set_ylabel("f [mHz]")
ax[0].set_title("Anchor track (m,k,n)=(2,0,0)")
ax[1].plot(t_days, fdot_obs, "C1.-"); ax[1].set_ylabel(r"$\dot f$ [Hz/s]")
ax[1].set_xlabel("Time [days]"); plt.tight_layout(); plt.show()

## 3. Plunge-time anchor

Backward (plunge-anchored) integration is more stable than forward integration:
it trades the uncertain $(p_0,e_0)$ for the well-measured plunge time. From the
observed $(f,\dot f)$ we get a Peters chirp-time estimate $\hat T_{\rm plunge}$,
which narrows the $T_{\rm plunge}$ prior to $\hat T_{\rm plunge}\pm$ a small
window — directly shrinking the search volume.

In [ ]:
T_plunge_est, T_plunge_per_seg = estimate_plunge_time(f_obs, fdot_obs)
print(f"True  T_plunge : {true_values[3]:.4f} yr")
print(f"Estim T_plunge : {T_plunge_est:.4f} yr  "
      f"(error {abs(T_plunge_est-true_values[3])*365.25:.1f} d)")

## 4. The differentiable forward model and loss (`fewtrax`)

For a Teukolsky harmonic $(m,k,n)$ along the adiabatic inspiral
$\theta=(M,\mu,a,p_0,e_0)$, the predicted instantaneous frequency is

$$
f_{mkn}(t;\theta)=\frac{1}{2\pi M_s}\bigl|\,m\,\Omega_\phi+k\,\Omega_\theta
+n\,\Omega_r\,\bigr|\Big|_{(p(t;\theta),\,e(t;\theta),\,a)} ,
$$

where $\Omega_{\phi,\theta,r}$ are the Kerr fundamental frequencies and
$(p(t),e(t))$ is the inspiral trajectory. With Gaussian STFT-bin errors the
log-likelihood of the track is $-\tfrac12 L(\theta)$ with the **track-MSE loss**

$$
\boxed{\;L(\theta)=\sum_\alpha
\frac{\bigl[f_{mkn}(t_\alpha;\theta)-\hat f(t_\alpha)\bigr]^2}{\sigma_f^2},
\qquad \sigma_f\approx \frac{1}{T_{\rm sft}}\;}
$$

In `emrisearch`, `TrackOptimizerJAX._build_loss(mode)` returns exactly this
$L(\theta)$ as a jitted closure. Internally it calls `fewtrax.EMRIInspiral` in
**backward mode**, evaluates `get_fundamental_frequencies` along the shared
$(p,e)$ trajectory, and interpolates to the SFT times via
$\tau=T_{\rm plunge}-t$:

```python
t_back, p_back, e_back, *_ = traj(p0=10.0, e0=e_f, T=T_plunge, a=a, M=M, mu=mu,
                                  backward=True, e_f=e_f, dense_steps=...)
f_track = jax.vmap(freq_one)(p_back, e_back)          # f_{mkn} along trajectory
f_pred  = jnp.interp(T_plunge*YEAR_SI - t_obs, t_back, f_track)
L       = jnp.sum((f_pred - f_obs)**2)
```

Because the whole thing is JAX-traced through the `diffrax` ODE adjoint,
$\nabla_\theta L$ and $\nabla^2_\theta L$ are available for free (~5–10 ms each).
Crucially, **one trajectory integration serves all harmonics**
(`_build_joint_loss_raw`), so the multi-mode loss costs essentially the same as
the single-mode one.

> **Setup.** Point `FEWTRAX_SRC` at the `fewtrax/src` tree and set the
> `FEW_DATA_DIR` environment variable to the flux-data directory. If unavailable,
> the cell is skipped and the rest of the notebook documents the intended flow.

In [ ]:
FEWTRAX_SRC = "/Users/bertd/Documents/PhD/LISA/Projects/JAX-waveform/fewtrax/src"
FEW_DATA_DIR = os.environ.get("FEW_DATA_DIR")

HAVE_FEWTRAX = False
flux_data = None
if os.path.isdir(FEWTRAX_SRC) and FEW_DATA_DIR:
    sys.path.insert(0, FEWTRAX_SRC)
    try:
        from fewtrax.data.loader import load_flux_data
        flux_data = load_flux_data(FEW_DATA_DIR)
        HAVE_FEWTRAX = True
        print("fewtrax flux data loaded.")
    except Exception as exc:
        print("fewtrax unavailable:", exc)
else:
    print("Set FEW_DATA_DIR (and FEWTRAX_SRC) to enable the fewtrax stages.")

In [ ]:
if HAVE_FEWTRAX:
    jax_opt = TrackOptimizerJAX(f_obs, fdot_obs, t_obs, flux_data, T_sft=T_sft)
    print(f"Narrowed T_plunge bounds: "
          f"[{jax_opt.bounds[3,0]:.3f}, {jax_opt.bounds[3,1]:.3f}] yr")

    # The differentiable loss L(theta) and its gradient at the truth.
    loss_fn = jax_opt._build_loss((m, 0, n))
    theta_true = jnp.array([true_values[0], true_values[1], true_values[2],
                            true_values[3], true_values[4]])
    L0   = float(loss_fn(theta_true))
    gradL = np.array(jax.grad(loss_fn)(theta_true))
    print(f"L(theta_true)          = {L0:.4e}")
    print(f"grad L (M,mu,a,Tpl,ef) = {gradL}")

## 5. Local fit: Adam → L-BFGS, then Fisher

A *local* optimiser polishes one basin to its MAP. `optimize_full` runs
multi-start Adam (robust to plateaus) followed by L-BFGS (superlinear final
convergence), both in unconstrained coordinates
($\log M,\log\mu,\tanh^{-1}a,\dots$) so steps never hit the bounds.

The local geometry is the **Fisher information** pulled back through the track
map. With residual Jacobian $J_{\alpha i}=\partial f_{mkn}(t_\alpha)/\partial\theta_i$,

$$
\Gamma_{\rm EMRI}=J^{\mathsf T}\Gamma_f J=\frac{J^{\mathsf T}J}{\sigma_f^2},
\qquad \Sigma_{\rm EMRI}\approx\Gamma_{\rm EMRI}^{-1}\;\;(\text{Cramér–Rao}).
$$

`compute_fisher` builds $J$ with `jax.jacobian` of the same fewtrax track. The
$1\sigma$ widths $\sqrt{\mathrm{diag}\,\Sigma}$ become the narrow prior for final
PE. (Caveat, Chua–Cutler: Fisher errors are only meaningful at the *true* basin —
which is why §6–§7 are needed first.)

In [ ]:
if HAVE_FEWTRAX:
    t0 = time.perf_counter()
    theta_map, loss_map = jax_opt.optimize_full(
        mode=(m, 0, n), n_starts=8, adam_steps=200, lbfgs_iter=100)
    print(f"Local fit in {time.perf_counter()-t0:.1f} s   loss = {loss_map:.3e}")
    print(f"MAP  : M={theta_map[0]:.3e} mu={theta_map[1]:.2f} a={theta_map[2]:.3f} "
          f"Tpl={theta_map[3]:.4f} ef={theta_map[4]:.4f}")
    print(f"True : M={true_values[0]:.3e} mu={true_values[1]:.2f} a={true_values[2]:.3f} "
          f"Tpl={true_values[3]:.4f} ef={true_values[4]:.4f}")

    fisher = jax_opt.compute_fisher(theta_map, mode=(m, 0, n))
    cov = np.linalg.inv(fisher)
    sigma = np.sqrt(np.maximum(np.diag(cov), 0.0))
    for name, val, s in zip(["M","mu","a","T_plunge","e_f"], theta_map, sigma):
        print(f"  {name:9s} = {val:.5g}  +/- {s:.3g}")

## 6. Global search with `parismc-jax` (PARIS)

### Why a global sampler is mandatory

The identification landscape is **structurally multimodal**. Chua & Cutler show
that around any injection there are tens-to-hundreds of *non-local secondaries*
that match the dominant frequency and its first two time-derivatives,

$$
\Bigl(\tfrac{d}{dt}\Bigr)^{k}\omega_\phi(t_0;\theta_{\rm sec})\approx
\Bigl(\tfrac{d}{dt}\Bigr)^{k}\omega_\phi(t_0;\theta_{\rm inj}),\quad k=0,1,2,
$$

but mismatch $\omega_\theta,\omega_r$ at $O(1)$. The single-track loss is exactly
the multi-time generalisation of this root system, so the multimodality is
*structural*, not an artefact — no single optimiser suffices.

### PARIS target

We sample the tempered posterior on the (already $T_{\rm plunge}$-narrowed) box,

$$
\ln P(\theta) = -\tfrac12 L(\theta) + \ln\pi(\theta),
$$

with $\pi$ uniform. PARIS runs $N_{\rm seed}$ parallel ARIS processes, each with a
Gaussian-mixture proposal centred on its own past *weighted* samples:

$$
q_t(x)\propto\sum_{t'<t} w_{t'}\,\tilde{\mathcal N}(x\mid x_{t'},\Sigma_{t-1}),
\qquad w_{t'}=\frac{P(x_{t'})}{q_{\rm total}(x_{t'})} .
$$

High-posterior samples become more probable proposal centres, so each process
locks onto a mode; redundant processes are merged, leaving **one process per
basin**. The modified Gaussian $\tilde{\mathcal N}$ applies the $e^{-p/2}$
self-density correction so newly discovered modes are not silently down-weighted.

### Wiring it to the fewtrax loss

PARIS works on the unit cube $[0,1]^5$ and calls `log_density` on cube points;
`prior_transform` maps the cube to the physical box (and is applied to the
returned samples). So `log_density(U)` does the affine map internally and reuses
the **same jitted `_build_loss`** from §4 — about 30 lines of glue.

In [ ]:
PARISMC_DIR = "/Users/bertd/Documents/PhD/LISA/Projects/JAX-waveform/parismc-jax"
HAVE_PARIS = False
if os.path.isdir(PARISMC_DIR):
    sys.path.insert(0, PARISMC_DIR)
    try:
        from parismc.sampler_jax import JaxSampler, JaxSamplerConfig
        HAVE_PARIS = True
        print("parismc (JaxSampler) available.")
    except Exception as exc:
        print("parismc unavailable:", exc)
else:
    print("parismc-jax not found at", PARISMC_DIR)

In [ ]:
def build_paris_target(optimizer, mode, sigma_f=None):
    """Wrap a TrackOptimizerJAX loss as a PARIS log-density on the unit cube.

    Returns (log_density, prior_transform):
      - log_density(U): U in [0,1]^5 -> ln P(theta) = -0.5 * L(theta)/sigma_f^2
      - prior_transform(U): [0,1]^5 -> physical box (applied to returned samples)
    """
    lo = jnp.array(optimizer.bounds[:, 0]); hi = jnp.array(optimizer.bounds[:, 1])
    loss_phys = optimizer._build_loss(mode)                 # reuse the fewtrax loss
    s2 = (1.0 / optimizer.T_sft) ** 2 if sigma_f is None else sigma_f ** 2

    prior_transform = lambda u: lo + (hi - lo) * u
    @jax.jit
    def log_density(U):                                     # U: (batch, 5)
        theta = lo + (hi - lo) * U
        L = jax.vmap(loss_phys)(theta)
        return -0.5 * L / s2
    return log_density, prior_transform

In [ ]:
if HAVE_FEWTRAX and HAVE_PARIS:
    log_density, prior_transform = build_paris_target(jax_opt, (m, 0, n))

    ndim, n_proc, n_iter = 5, 32, 2000
    cfg = JaxSamplerConfig(n_proc_max=n_proc, n_max=n_iter + 16,
                           alpha=1000, gamma=100, seed=0)
    smp = JaxSampler(ndim=ndim, log_density_func=log_density,
                     config=cfg, prior_transform=prior_transform)

    # Seed from the top-N_proc Latin-hypercube points on the cube.
    rng = np.random.default_rng(0)
    U_lhs   = rng.random((5000, ndim))
    ld_lhs  = np.asarray(log_density(jnp.array(U_lhs)))
    top     = np.argsort(ld_lhs)[-n_proc:]
    init_cov = (0.05 ** 2) * np.eye(ndim)

    smp.state = smp.init_state(seed_points=U_lhs[top],
                               seed_logdens=ld_lhs[top], init_cov=init_cov)
    t0 = time.perf_counter()
    smp.run(num_iterations=n_iter, verbose=True)            # uses self.state
    print(f"PARIS finished in {time.perf_counter()-t0:.1f} s")

    # Basin registry: samples already in PHYSICAL units (prior_transform applied).
    samples, weights = smp.get_samples_with_weights()
    print(f"registry: {samples.shape[0]} weighted samples, logZ = {smp.log_evidence():.3f}")
else:
    samples, weights = None, None
    print("PARIS stage skipped (needs fewtrax + parismc).")

### The basin registry

PARIS returns weighted samples in physical space. Clustering them gives the
basin registry $\{\hat\theta_j, w_j, \Sigma_j\}$ — the candidate modes, their
posterior masses, and local covariances. After round 0 (the dominant track
only), most basins are *spurious* secondaries; §7 removes them with higher-mode
information. Below we just show the marginal weighted posterior for a couple of
parameters as a sanity check.

In [ ]:
if samples is not None and len(samples) > 0:
    w = weights / weights.sum()
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, idx, name in zip(axes, [0, 2, 3], ["M", "a", "T_plunge"]):
        ax.hist(samples[:, idx], bins=40, weights=w, color="C0", alpha=0.8)
        ax.axvline(true_values[[0,1,2,3,4][idx]], color="k", ls="--")
        ax.set_xlabel(name)
    axes[0].set_ylabel("weighted density"); plt.tight_layout(); plt.show()
else:
    print("No registry to plot (PARIS stage skipped).")

## 7. Iterative higher-mode refinement of the posterior

Tracks constrain only the *geometry* ($\mu/M$ and, via ratios, $a$); the spurious
secondaries match $\omega_\phi$ but mismatch $\omega_\theta,\omega_r$. Adding a
**companion harmonic** that depends on $\omega_r$ imposes an independent root
system and collapses them. PARIS's importance weights make this a pure
**reweighting** — no re-run:

$$
\ln w^{\rm new}=\ln w^{\rm old}-\tfrac12 L_2(\theta),
\qquad P\;\to\;P\cdot e^{-L_2/2},
$$

where $L_2$ is the track-MSE loss of the new mode. The decision of *which*
companion modes are real is made with the **kept detection statistic**: predict
the secondary track from the current $\hat\theta$, and confirm it if
$\Lambda>\Lambda_{\rm thr}$. This is exactly what
`IterativeTrackSearch._predict_and_vet` / `_vet_track` already do.

Round 0 → top basins → predict $(3,0,0)$, $(2,0,1)$, … → vet with `det_stat` →
multiply the registry weights by $e^{-L_2/2}$ → secondaries that matched only
$\omega_\phi$ now carry a large $L_2$ and their weight vanishes.

In [ ]:
# Illustration of the vetting half (the kept statistic) for candidate companions.
candidates = [(2, 0, 1), (3, 0, 0), (4, 0, 0), (2, 0, -1)]
print("Companion-mode vetting via the KEPT detection statistic (true-track proxy):")
for (mm, kk, nn) in candidates:
    f_c    = mm * f[0] + nn * f[1]          # n labels the radial harmonic here
    fdot_c = mm * dotf[0] + nn * dotf[1]
    valid  = np.isfinite(f_c) & (f_c > 0)
    if valid.sum() < 3:
        print(f"  mode {(mm,kk,nn)}: out of band"); continue
    Lam = float(jax_det_stat(data_sfts, jnp.asarray(f_c), jnp.asarray(fdot_c), T_sft=T_sft))
    print(f"  mode {(mm,kk,nn)}: Lambda = {Lam:8.1f}")

In [ ]:
def reweight_registry(samples, weights, optimizer, new_mode, sigma_f=None):
    """Fold a confirmed companion harmonic into the registry: P -> P * exp(-L2/2).

    No resampling: evaluate L2 at the stored samples and multiply the weights.
    """
    s2 = (1.0 / optimizer.T_sft) ** 2 if sigma_f is None else sigma_f ** 2
    loss2 = optimizer._build_loss(new_mode)                 # L2(theta) for new mode
    L2 = np.asarray(jax.vmap(loss2)(jnp.asarray(samples)))
    return weights * np.exp(-0.5 * L2 / s2)

if (samples is not None) and HAVE_FEWTRAX:
    # Example: a (2,0,1) companion is confirmed -> reweight the round-0 registry.
    w_new = reweight_registry(samples, weights, jax_opt, (2, 0, 1))
    ess0 = weights.sum()**2 / (weights**2).sum()
    ess1 = w_new.sum()**2 / (w_new**2).sum()
    print(f"effective sample size: round0 = {ess0:.0f} -> round1 = {ess1:.0f}")
    print("(spurious basins lose weight; surviving basins dominate)")
else:
    print("Reweighting demo skipped (needs the PARIS registry).")

### Full driver

The whole Round-0 → vet → reweight loop is packaged in
`emrisearch.IterativeTrackSearch`, which today carries a single MAP; the upgrade
described in the report is to carry the **PARIS registry** and make the
higher-mode step a reweight (above) rather than a hard joint refit. The
`run(f_obs, fdot_obs, anchor_mode)` entry point returns a `SearchResult` with the
MAP, covariance, confirmed modes, and the narrow MCMC bounds.

In [ ]:
if HAVE_FEWTRAX:
    from emrisearch.iterative_track_search import IterativeTrackSearch
    its = IterativeTrackSearch(flux_data, data_sfts, t_obs, T_sft=T_sft,
                               vet_threshold=50.0, max_iterations=3,
                               adam_starts=8, adam_steps=150, lbfgs_iter=100)
    result = its.run(f_obs, fdot_obs, anchor_mode=(2, 0, 0))
    print(result.summary())
else:
    print("IterativeTrackSearch skipped (needs fewtrax).")

## 8. Handoff to final parameter estimation

The collapsed registry gives the true basin $\hat\theta$ with covariance
$\Sigma$. `MCMCHandoff` / `prepare_mcmc_handoff` convert
$(\hat\theta,\Sigma,T_{\rm plunge})$ into FEW initial conditions $(p_0,e_0)$ and a
narrow $n\sigma$ prior; `FastEMRILikelihood` is the FEW + LISA-response
likelihood for the final sampler (seed a blackjax-NUTS chain in the Fisher
ellipsoid, or eryn if residual multimodality persists). The higher-payoff
alternative is the **coherent WDM likelihood**, which additionally carries
amplitude/phase and so breaks the residual $M$–$\mu$ degeneracy that tracks alone
cannot.

$$
\ln\mathcal L_{\rm WDM}(\theta)=-\tfrac12\sum_{n,q}
\frac{|d_{nq}-W_{nq}(\theta)|^2}{S_n(q\Delta F)} .
$$


In [ ]:
if HAVE_FEWTRAX and 'result' in dir():
    from emrisearch.mcmc_likelihood import prepare_mcmc_handoff
    try:
        handoff = prepare_mcmc_handoff(result)
        print(handoff.summary())
    except Exception as exc:
        print("Handoff illustration skipped:", exc)
else:
    print("Handoff skipped (needs a SearchResult from §7).")

## Summary

| Stage | Tool | Status |
|---|---|---|
| Detection statistic $\Lambda$ | `jax_utils.det_stat` (**kept**) | trigger + secondary-mode vetting |
| Track loss $L(\theta)$, grads, Fisher | `fewtrax` via `TrackOptimizerJAX` | exists |
| Local fit Adam→L-BFGS | `optimize_full` / `compute_fisher` | exists |
| Global multimodal search | `parismc-jax` `JaxSampler` | ~30-line adapter (`build_paris_target`) |
| Higher-mode refinement | reweight registry $P\!\to\!P e^{-L_2/2}$ | `reweight_registry` + `IterativeTrackSearch` |
| PE handoff | `MCMCHandoff` / `FastEMRILikelihood` | exists |

**Key idea:** keep the detection statistic; replace the SVD/DE identification
with a differentiable `fewtrax` loss, a PARIS global sampler that respects the
Chua–Cutler multimodality, and an iterative higher-mode reweighting that sharpens
the posterior with no re-run.
